In [2]:
!pip install datasets

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl (15 kB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
   ---------------------------------------- 0.0/27.5 MB ? eta -:--:--
   -- ------------------------------------- 1.6/27.5 MB 8.0 MB/s eta 0:00:04
   ---- ----------------------------------- 2.9/27.5 MB 7.3 MB/s eta 0:00:04
   ------- -------------------------------- 5.2/27.5 MB 8.1 MB/s eta 0:00:03
   --------- ------------------------------ 6.6/27.5 MB 7.7 MB/s eta 0:00:03
   ------------ --------------------------- 8.7/27.5 MB 8.1 MB/s eta 0:00:03
   -------------- ------------------------- 10.2/27.5 MB 8.4 MB/s eta 0:00:03
   ----------------- ---------------------- 11.8/27.5 MB 7.9 MB/s eta 0:00:02
   -----

In [ ]:
# File naming convention

# Each of the 1440 files has a unique filename. The filename consists of a 7-part numerical identifier (e.g., 03-01-06-01-02-01-12.wav). These identifiers define the stimulus characteristics:

# Filename identifiers

# Modality (01 = full-AV, 02 = video-only, 03 = audio-only).

# Vocal channel (01 = speech, 02 = song).

# Emotion (01 = neutral, 02 = calm, 03 = happy, 04 = sad, 05 = angry, 06 = fearful, 07 = disgust, 08 = surprised).

# Emotional intensity (01 = normal, 02 = strong). NOTE: There is no strong intensity for the 'neutral' emotion.

# Statement (01 = "Kids are talking by the door", 02 = "Dogs are sitting by the door").

# Repetition (01 = 1st repetition, 02 = 2nd repetition).

# Actor (01 to 24. Odd numbered actors are male, even numbered actors are female).

# Filename example: 03-01-06-01-02-01-12.wav

# Audio-only (03)
# Speech (01)
# Fearful (06)
# Normal intensity (01)
# Statement "dogs" (02)
# 1st Repetition (01)
# 12th Actor (12)
# Female, as the actor ID number is even.

In [12]:
!pip install soundfile librosa

In [14]:
from datasets import load_dataset, load_from_disk

# 1. Load the dataset from the Hub
dataset = load_dataset("ehzawad/sinhala-emotion-dataset")

# 2. Save it to a local folder named "my_sinhala_data"
dataset.save_to_disk("my_sinhala_data")

print("Dataset saved successfully!")

# --- Later, to load it back from your local folder ---
# This is much faster and works offline
reloaded_dataset = load_from_disk("my_sinhala_data")
print(reloaded_dataset)

Repo card metadata block was not found. Setting CardData to empty.


Saving the dataset (0/3 shards):   0%|          | 0/2999 [00:00<?, ? examples/s]

Dataset saved successfully!
DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'sampling_rate', 'id'],
        num_rows: 2999
    })
})


In [ ]:
import soundfile as sf
from datasets import load_from_disk
import os
import pandas as pd
from tqdm import tqdm  # Progress bar

# 1. Load the dataset from your local folder
dataset = load_from_disk("my_sinhala_data")

# 2. Create a folder to store the extracted audio
output_folder = "extracted_sinhala_dataset"
audio_dir = os.path.join(output_folder, "audio_files")
os.makedirs(audio_dir, exist_ok=True)

# 3. Function to process a single split (e.g., 'train', 'test')
def extract_split(split_name, data_split):
    print(f"Processing {split_name}...")
    metadata = []
    
    for i, item in enumerate(tqdm(data_split)):
        # Construct a filename
        filename = f"{split_name}_{i}.wav"
        file_path = os.path.join(audio_dir, filename)
        
        # Get audio data
        audio_array = item['audio']['array']
        sample_rate = item['audio']['sampling_rate']
        
        # Save the audio file
        sf.write(file_path, audio_array, sample_rate)
        
        # Save the text/label info to our list
        # Adapting to your specific dataset columns (text, emotion, etc.)
        record = {
            "audio_filename": filename,
            "text": item.get('text', ''),
            "label": item.get('label', ''), # specific label key might vary
            "original_split": split_name
        }
        metadata.append(record)
        
    return metadata

# 4. Iterate through all splits (train, test, etc.)
all_metadata = []

# Check if dataset is a Dict (has splits) or just one Dataset
if hasattr(dataset, 'keys'):
    keys = dataset.keys() # ['train', 'test', etc.]
else:
    keys = ['data']
    dataset = {'data': dataset}

for split in keys:
    split_data = extract_split(split, dataset[split])
    all_metadata.extend(split_data)

# 5. Save the metadata to a CSV file (Excel readable)
df = pd.DataFrame(all_metadata)
df.to_csv(os.path.join(output_folder, "metadata.csv"), index=False)

print(f"\nDone! Files extracted to: {output_folder}")

In [ ]:
import soundfile as sf
import numpy as np

# Get audio data
audio_array = sample['audio']['array']
sampling_rate = sample['audio']['sampling_rate']

# Save audio to file
sf.write('sample_audio.wav', audio_array, sampling_rate)

# Basic audio analysis
duration = len(audio_array) / sampling_rate
print(f"Audio duration: {duration:.2f} seconds")

In [7]:
# Already trained whisper
# Requires: librosa
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
import librosa
import torch
import numpy as np

model_id = "firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3"
model = AutoModelForAudioClassification.from_pretrained(model_id)

feature_extractor = AutoFeatureExtractor.from_pretrained(model_id, do_normalize=True)
id2label = model.config.id2label


Loading weights:   0%|          | 0/491 [00:00<?, ?it/s]

In [4]:
def preprocess_audio(audio_path, feature_extractor, max_duration=30.0):
    audio_array, sampling_rate = librosa.load(audio_path, sr=None)
    
    max_length = int(feature_extractor.sampling_rate * max_duration)
    if len(audio_array) > max_length:
        audio_array = audio_array[:max_length]
    else:
        audio_array = np.pad(audio_array, (0, max_length - len(audio_array)))

    inputs = feature_extractor(
        audio_array,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=max_length,
        truncation=True,
        return_tensors="pt",
    )
    return inputs

In [5]:
def predict_emotion(audio_path, model, feature_extractor, id2label, max_duration=30.0):
    inputs = preprocess_audio(audio_path, feature_extractor, max_duration)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_id = torch.argmax(logits, dim=-1).item()
    predicted_label = id2label[predicted_id]
    
    return predicted_label

In [9]:
from moviepy import AudioFileClip
import os

def convert_m4a_to_wav_moviepy(input_path, output_path):
    if not os.path.exists(input_path):
        print(f"Error: File not found at {input_path}")
        return

    try:
        # Load the m4a file
        audio = AudioFileClip(input_path)
        
        # Write to wav
        # Note: In MoviePy v2, explicit logger=None prevents spamming the console if you prefer
        audio.write_audiofile(output_path, codec='pcm_s16le', fps=44100)
        
        # Close the clip
        audio.close()
        print(f"\nSuccess! Saved to: {output_path}")
        
    except Exception as e:
        print(f"Conversion failed: {e}")

# Your paths
input_file = r'E:\Extra_Work\SLITT\MySolution\AI\Training\Emotion_Voice_Datasets\Recording.m4a'
output_file = 'output_song.wav'

convert_m4a_to_wav_moviepy(input_file, output_file)

MoviePy - Writing audio in output_song.wav


MoviePy - Done.

Success! Saved to: output_song.wav


In [10]:
audio_path = r"E:\Extra_Work\SLITT\MySolution\AI\Training\output_song.wav"

predicted_emotion = predict_emotion(audio_path, model, feature_extractor, id2label)
print(f"Predicted Emotion: {predicted_emotion}")

Predicted Emotion: sad
